[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ivastar/ml_for_astro_notebooks/blob/main/session1a_solutions.ipynb)

# Session 1 — Regression as a Machine Learning Method

In this session we take the most familiar tool in the astronomer's toolbox — fitting a straight line — and look at it through the lens of machine learning. Along the way we will meet the ideas that every ML method relies on:

1. **Data** - generate some data that we can use
2. **Least squares** — fitting a line by hand, with and without measurement uncertainties
3. **Linear regression** — the same fit, done the "ML way" with `scikit-learn`
4. **Train/test splits** — how do we know a model works on data it has never seen?
5. **Overfitting** — what happens when the model is too flexible for the data
6. **Regularization** — Ridge and Lasso as a way to keep flexible models in check

Cells marked **✏️ Exercise** are for you to complete. Everything else can be run as is.

In [ ]:
import warnings
import numpy as np
import matplotlib.pyplot as plt

from sklearn.linear_model import LinearRegression, Ridge, Lasso, RidgeCV, LassoCV
from sklearn.model_selection import train_test_split, KFold, cross_val_score
from sklearn.preprocessing import PolynomialFeatures, StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.exceptions import ConvergenceWarning

plt.rcParams.update({
    "figure.figsize": (7.5, 4.5), "figure.dpi": 110,
    "font.size": 12, "axes.titlesize": 13,
    "axes.spines.top": False, "axes.spines.right": False,
    "axes.grid": True, "grid.alpha": 0.25, "grid.linewidth": 0.8,
    "lines.linewidth": 2, "legend.frameon": False,
})

## 1. The data

We generate a simple linear relation $y = 3x + 3$ and add noise. The noise is drawn from a **Poisson distribution** whose expectation value is the model value itself, centred on zero and scaled by a factor of 2.

Since a Poisson variable with mean $\lambda$ has variance $\lambda$, the scatter on each point is

$$\sigma_i = 2\sqrt{3x_i + 3},$$

so the uncertainties **grow with $x$**. Data like this, where the noise is not the same for every point, is called *heteroscedastic*. Keep that in mind — it matters for which fit is "best".

 **✏️ Exercise 0** code up the generation of this dataset:

In [ ]:
np.random.seed(16)                    # set seed for reproducibility

# YOUR CODE HERE, should define N number of points, x as the independent variable, y_true as the true model, y as the observed data with Poisson scatter, and sigma as the uncertainty of each point

### BEGIN SOLUTION
N = 100
x = np.arange(N)
y_true = 3*x + 3                      # the underlying linear model
y = y_true + 2*(np.random.poisson(y_true, N) - y_true)   # Poisson scatter centred on 0
sigma = 2*np.sqrt(y_true)             # the known 1-sigma uncertainty of each point
### END SOLUTION

fig, ax = plt.subplots()
ax.errorbar(x, y, yerr=sigma, fmt="o", ms=4, color="gray", ecolor="gray",
            elinewidth=1, label="data")
ax.plot(x, y_true, color="black", ls="--", label="true model  $y = 3x + 3$")
ax.set(xlabel="x", ylabel="y", title="Simulated data with Poisson-like scatter")
ax.legend();

## 2. Least squares by hand

We want the straight line $y = a + b\,x$ that best describes the data. "Best" needs a definition: least squares picks the parameters that minimise the sum of squared residuals

$$S(a, b) = \sum_{i=1}^{N} \left(y_i - a - b\,x_i\right)^2 .$$

It is convenient to write this in matrix form. Collect the parameters in a vector $\boldsymbol\theta = (a, b)^T$ and build the **design matrix** $A$, whose rows are $(1, x_i)$:

$$A = \begin{pmatrix} 1 & x_1 \\ 1 & x_2 \\ \vdots & \vdots \\ 1 & x_N \end{pmatrix}, \qquad \text{model} = A\,\boldsymbol\theta .$$

Setting the derivative of $S = (\mathbf{y} - A\boldsymbol\theta)^T(\mathbf{y} - A\boldsymbol\theta)$ to zero gives the **normal equations**

$$A^T A \,\boldsymbol\theta = A^T \mathbf{y} \quad\Longrightarrow\quad \boldsymbol\theta = (A^T A)^{-1} A^T \mathbf{y}.$$

Notice that the model only needs to be linear in the **parameters**, not in $x$. Any column you can compute from $x$ (e.g. $x^2$, $\sin x$, $\log x$) can go into $A$ — we will use exactly this trick in Section 5.

### ✏️ Exercise 1 — Ordinary least squares

Build the design matrix `A` and solve the normal equations for `theta_ols = [a, b]`.

*Hint:* `np.column_stack` builds `A`. Prefer `np.linalg.solve(M, v)` over computing `np.linalg.inv(M) @ v` — it is faster and numerically more stable.

In [ ]:
### BEGIN SOLUTION
A = np.column_stack([np.ones(N), x])
theta_ols = np.linalg.solve(A.T @ A, A.T @ y)
### END SOLUTION

a_ols, b_ols = theta_ols
print(f"OLS:  intercept a = {a_ols:6.2f},  slope b = {b_ols:.3f}")

In [ ]:
# Check your answer against numpy's built-in routines
print("np.polyfit      :", np.polyfit(x, y, 1)[::-1])        # polyfit returns [b, a]
print("np.linalg.lstsq :", np.linalg.lstsq(A, y, rcond=None)[0])
assert np.allclose(theta_ols, np.polyfit(x, y, 1)[::-1]), "Not quite — check A and the normal equations"

### Weighted least squares and $\chi^2$

Ordinary least squares treats every point as equally trustworthy. Our points are not: the scatter at $x = 90$ is more than five times the scatter at $x = 1$. If we know the uncertainties, the natural thing to minimise is

$$\chi^2 = \sum_{i=1}^{N} \left(\frac{y_i - a - b\,x_i}{\sigma_i}\right)^2 ,$$

which is the maximum-likelihood solution for Gaussian errors. With the weight matrix $W = \mathrm{diag}(1/\sigma_i^2)$ the solution becomes

$$\boldsymbol\theta = (A^T W A)^{-1} A^T W\,\mathbf{y}, \qquad \mathrm{Cov}(\boldsymbol\theta) = (A^T W A)^{-1}.$$

We get the **parameter uncertainties for free** from the covariance matrix.

### ✏️ Exercise 2 — Weighted least squares

Compute the covariance matrix `cov_wls` and best-fit parameters `theta_wls`. The cell then prints the parameter uncertainties and the reduced $\chi^2$.

In [ ]:
### BEGIN SOLUTION
W = np.diag(1/sigma**2)
cov_wls = np.linalg.inv(A.T @ W @ A)
theta_wls = cov_wls @ A.T @ W @ y
### END SOLUTION

err_wls = np.sqrt(np.diag(cov_wls))
chi2 = np.sum(((y - A @ theta_wls) / sigma)**2)
dof = N - len(theta_wls)
print(f"WLS:  a = {theta_wls[0]:.2f} ± {err_wls[0]:.2f},  b = {theta_wls[1]:.3f} ± {err_wls[1]:.3f}")
print(f"chi2 / dof = {chi2:.1f} / {dof} = {chi2/dof:.2f}")

In [ ]:
# Check: np.polyfit takes weights w = 1/sigma (not 1/sigma^2!)
p, cov = np.polyfit(x, y, 1, w=1/sigma, cov="unscaled")
print("np.polyfit (weighted):", p[::-1], "±", np.sqrt(np.diag(cov))[::-1])
assert np.allclose(theta_wls, p[::-1])

In [ ]:
fig, (ax, axr) = plt.subplots(2, 1, figsize=(7.5, 6), sharex=True,
                              gridspec_kw=dict(height_ratios=[3, 1.3], hspace=0.08))
ax.errorbar(x, y, yerr=sigma, fmt="o", ms=4, color="gray", ecolor="gray",
            elinewidth=1, label="data")
ax.plot(x, y_true, color="black", ls="--", lw=1.5, label="true model")
ax.plot(x, A @ theta_ols, label=f"OLS:  {a_ols:.1f} + {b_ols:.2f}x")
ax.plot(x, A @ theta_wls, label=f"WLS:  {theta_wls[0]:.1f} + {theta_wls[1]:.2f}x")
ax.set(ylabel="y", title="Ordinary vs. weighted least squares")
ax.legend()

axr.axhline(0, color="black", lw=1)
axr.plot(x, (y - A @ theta_wls) / sigma, "o", ms=4, color="C1")
axr.set(xlabel="x", ylabel=r"(y − model) / σ")
axr.axhspan(-1, 1, color="gray", alpha=0.12, lw=0);

## 3. The same fit as a machine learning model

Machine learning libraries describe exactly the same problem with different words:

| Astronomy / statistics | Machine learning | In our example |
|---|---|---|
| independent variable | **feature(s)**, `X` | `x` |
| dependent variable | **target** / label, `y` | `y` |
| model parameters | **weights** (`coef_`) and **bias** (`intercept_`) | $b$, $a$ |
| $\chi^2$ / sum of squared residuals | **loss function** | $S(a,b)$ |
| fitting | **training** | `.fit()` |
| evaluating the model | **predicting** / inference | `.predict()` |

In `scikit-learn`, every model follows the same pattern: create it, `.fit(X, y)`, then `.predict(X_new)`. Features are always a 2D array of shape `(n_samples, n_features)` — even when there is only one feature.

### ✏️ Exercise 3 — `LinearRegression`

1. Reshape `x` into the feature matrix `X` of shape `(100, 1)`.
2. Fit an unweighted `LinearRegression` (call it `lr`) and a weighted one (`lr_w`, use the `sample_weight` argument of `.fit`).
3. Use `lr_w` to predict `y` at $x = 150$ and $x = 500$.

In [ ]:
### BEGIN SOLUTION
X = x.reshape(-1, 1) # -1 is a placeholder that avoids the need to manually specify the number of rows
lr = LinearRegression().fit(X, y)
lr_w = LinearRegression().fit(X, y, sample_weight=1/sigma**2)
y_new = lr_w.predict(np.array([[150], [500]]))
### END SOLUTION

print(f"sklearn unweighted: a = {lr.intercept_:.2f}, b = {lr.coef_[0]:.3f}   (OLS by hand: {a_ols:.2f}, {b_ols:.3f})")
print(f"sklearn weighted  : a = {lr_w.intercept_:.2f}, b = {lr_w.coef_[0]:.3f}   (WLS by hand: {theta_wls[0]:.2f}, {theta_wls[1]:.3f})")
print(f"prediction at x=150: {y_new[0]:.1f}   (true: {3*150+3})")
print(f"prediction at x=500: {y_new[1]:.1f}   (true: {3*500+3})")

## 4. Train/test split

The error of a model on the data it was trained on is an **optimistic** estimate of how it will do on new data: the model has already "seen" the noise in those points. The standard remedy is to hold back part of the data:

* **training set** — used to fit the model
* **test set** — used *only* to evaluate it, at the very end

Common metrics for regression are the root-mean-square error

$$\mathrm{RMSE} = \sqrt{\frac{1}{n}\sum_i (y_i - \hat y_i)^2}$$

and the coefficient of determination $R^2$ (1 = perfect, 0 = no better than predicting the mean).

### ✏️ Exercise 4 — Split, fit, evaluate

1. Use `train_test_split` to split `X`, `y` (and `sigma`, so we can keep track of it) into 70% training and 30% test data. Use `random_state=42`.
2. Fit a `LinearRegression` called `lr_tt` on the training set only.
3. Compute the RMSE and $R^2$ on both the training and the test set.

In [ ]:
from sklearn.metrics import mean_squared_error, r2_score

def rmse(y_true, y_pred):
    return np.sqrt(mean_squared_error(y_true, y_pred))

### BEGIN SOLUTION
X_train, X_test, y_train, y_test, s_train, s_test = train_test_split(
    X, y, sigma, test_size=0.3, random_state=42)
lr_tt = LinearRegression().fit(X_train, y_train)
rmse_train, rmse_test = rmse(y_train, lr_tt.predict(X_train)), rmse(y_test, lr_tt.predict(X_test))
r2_train, r2_test = r2_score(y_train, lr_tt.predict(X_train)), r2_score(y_test, lr_tt.predict(X_test))
### END SOLUTION

print(f"train: N = {len(y_train):3d}   RMSE = {rmse_train:6.2f}   R2 = {r2_train:.3f}")
print(f"test : N = {len(y_test):3d}   RMSE = {rmse_test:6.2f}   R2 = {r2_test:.3f}")

In [ ]:
fig, ax = plt.subplots()
ax.plot(X_train, y_train, "o", label=f"train (N={len(y_train)})")
ax.plot(X_test, y_test, "s", label=f"test (N={len(y_test)})")
xx = np.linspace(0, N, 200).reshape(-1, 1)
ax.plot(xx, lr_tt.predict(xx), color="black", lw=1.5, label="fit to training set")
ax.set(xlabel="x", ylabel="y", title="Train/test split")
ax.legend();

## 5. Overfitting

Let's give the model more freedom. **Polynomial regression** is still *linear* regression — we simply add columns $x^2, x^3, \dots, x^d$ to the design matrix (Section 2!). `PolynomialFeatures` does this for us.

We also make the problem more realistic: in astronomy, the **labelled** data (for example, galaxies with spectroscopic redshifts) is usually a small fraction of the full sample. So we train on only **15 points** and test on the remaining 85.

Two practical details:
* Powers of $x$ up to $x^{14}$ span ~28 orders of magnitude for $x \sim 100$, which makes the linear algebra numerically unstable. We therefore **standardise** $x$ (zero mean, unit variance) before taking powers, and standardise the resulting features again.
* `make_pipeline` chains these steps so that `.fit` and `.predict` apply all of them consistently to training and test data.

In [ ]:
# Small training set, large test set
X_tr, X_te, y_tr, y_te = train_test_split(X, y, train_size=15, random_state=9)

def poly_model(degree, regressor=None):
    # scale x -> polynomial features x, x^2, ..., x^d -> scale features -> regressor
    if regressor is None:
        regressor = LinearRegression()
    return make_pipeline(StandardScaler(),
                         PolynomialFeatures(degree, include_bias=False),
                         StandardScaler(),
                         regressor)

# Example: a degree-3 polynomial
m3 = poly_model(3).fit(X_tr, y_tr)
print(f"degree 3: train RMSE = {rmse(y_tr, m3.predict(X_tr)):.1f},  test RMSE = {rmse(y_te, m3.predict(X_te)):.1f}")


For each polynomial degree from 1 to 14, fit `poly_model(degree)` on the small training set and store the training and test RMSE in the lists `rmse_tr` and `rmse_te`. The next cell plots the result.

In [ ]:
degrees = np.arange(1, 15)
rmse_tr, rmse_te = [], []

for d in degrees:
    m = poly_model(d).fit(X_tr, y_tr)
    rmse_tr.append(rmse(y_tr, m.predict(X_tr)))
    rmse_te.append(rmse(y_te, m.predict(X_te)))

for d, a_, b_ in zip(degrees, rmse_tr, rmse_te):
    print(f"degree {d:2d}:  train RMSE = {a_:10.2f}   test RMSE = {b_:12.2f}")

In [ ]:
fig, ax = plt.subplots()
ax.plot(degrees, rmse_tr, "o-", label="training set")
ax.plot(degrees, rmse_te, "s-", label="test set")
ax.axhline(np.sqrt(np.mean(sigma**2)), color="gray", ls=":", lw=1.5, label="typical noise level")
ax.set_yscale("log")
ax.set(xlabel="polynomial degree", ylabel="RMSE", xticks=degrees,
       title="Training error keeps falling; test error does not")
ax.legend();

In [ ]:
xx = np.linspace(0, N - 1, 500).reshape(-1, 1)
fig, ax = plt.subplots()
ax.plot(X_te, y_te, "o", ms=4, color="gray", alpha=0.4, label="test data")
ax.plot(X_tr, y_tr, "o", ms=8, color="black", label="training data (15 points)")
for d in [1, 4, 9, 14]:
    ax.plot(xx, poly_model(d).fit(X_tr, y_tr).predict(xx), label=f"degree {d}")
ax.set(xlabel="x", ylabel="y", ylim=(-100, 450), title="Polynomial fits to a small training set")
ax.legend(ncol=2, fontsize=10, loc="upper left");

### Choosing the complexity without touching the test set

If we pick the degree by looking at the test error, the test set is no longer an independent check — we have used it for training decisions. The standard solution is **k-fold cross-validation** on the training set: split it into $k$ folds, train on $k-1$, validate on the remaining one, rotate, and average.

In [ ]:
cv = KFold(n_splits=5, shuffle=True, random_state=0)
cv_rmse = [-cross_val_score(poly_model(d), X_tr, y_tr, cv=cv,
                            scoring="neg_root_mean_squared_error").mean()
           for d in degrees]
best_degree = degrees[np.argmin(cv_rmse)]
for d, r in zip(degrees, cv_rmse):
    print(f"degree {d:2d}:  CV RMSE = {r:14.1f}")
print("degree chosen by cross-validation:", best_degree)

## 6. Regularization: Ridge and Lasso

Choosing a simpler model is one way to avoid overfitting. Another is to keep the flexible model but **penalise large coefficients**. We add a penalty term to the loss:

$$\textbf{Ridge (L2):}\quad \mathcal{L} = \sum_i \left(y_i - \hat y_i\right)^2 + \alpha \sum_j \theta_j^2$$

$$\textbf{Lasso (L1):}\quad \mathcal{L} = \frac{1}{2n}\sum_i \left(y_i - \hat y_i\right)^2 + \alpha \sum_j |\theta_j|$$

* $\alpha$ is a **hyperparameter**: it is not fitted from the data by `.fit`, we have to choose it. $\alpha = 0$ gives back ordinary least squares; $\alpha \to \infty$ forces all coefficients to zero.
* **Ridge** shrinks all coefficients smoothly towards zero.
* **Lasso** can set coefficients **exactly** to zero — it performs automatic **feature selection**.
* The penalty treats all coefficients equally, so the features must be on comparable scales. This is why our pipeline standardises the features. (The intercept is not penalised.)

*Bayesian aside:* Ridge is equivalent to a Gaussian prior on the coefficients, and Lasso to a Laplace prior — $\alpha$ sets the prior width.

In [ ]:
# Ridge with a degree-14 polynomial: increasing the penalty tames the fit
fig, ax = plt.subplots()
ax.plot(X_te, y_te, "o", ms=4, color="gray", alpha=0.4, label="test data")
ax.plot(X_tr, y_tr, "o", ms=8, color="black", label="training data")
for alpha in [1e-6, 1e-2, 1, 100]:
    m = poly_model(14, Ridge(alpha=alpha)).fit(X_tr, y_tr)
    ax.plot(xx, m.predict(xx),
            label=f"α = {alpha:g}  (test RMSE {rmse(y_te, m.predict(X_te)):.0f})")
ax.set(xlabel="x", ylabel="y", ylim=(-100, 450), title="Ridge regression, degree-14 polynomial")
ax.legend(fontsize=10, loc="upper left");

In [ ]:
alphas = np.logspace(-4, 3, 60)
warnings.filterwarnings("ignore", category=ConvergenceWarning)   # Lasso is slow to converge at tiny alpha

# Coefficient paths: how each coefficient changes with alpha

def coef_path(make_regressor):
    return np.array([poly_model(14, make_regressor(a)).fit(X_tr, y_tr)[-1].coef_ for a in alphas])

coefs_ridge = coef_path(lambda a: Ridge(alpha=a))
coefs_lasso = coef_path(lambda a: Lasso(alpha=a, max_iter=100_000))

# colour each coefficient by the power of x it multiplies (light = low power, dark = high)
ramp = plt.cm.Blues(np.linspace(0.35, 1.0, 14))
fig, axes = plt.subplots(1, 2, figsize=(11, 4.2))
for ax, coefs, name in zip(axes, [coefs_ridge, coefs_lasso], ["Ridge", "Lasso"]):
    for j in range(14):
        ax.plot(alphas, coefs[:, j], color=ramp[j], lw=1.6)
    ax.set_xscale("log")
    ax.set_yscale("symlog", linthresh=1)
    ax.set(xlabel="α", ylabel="coefficient (standardised features)", title=f"{name} coefficient path")
k = np.argmin(abs(alphas - 10))
axes[1].annotate("only $x^1$ survives", xy=(alphas[k], coefs_lasso[k, 0]), xytext=(-40, 25),
                 textcoords="offset points", color="black", fontsize=10,
                 arrowprops=dict(arrowstyle="-", color="black", lw=0.8))
axes[1].set_ylabel("")
sm = plt.cm.ScalarMappable(cmap=plt.cm.colors.ListedColormap(ramp),
                           norm=plt.Normalize(0.5, 14.5))
fig.colorbar(sm, ax=axes, label="power of x", ticks=[1, 4, 7, 10, 14])

print("non-zero Lasso coefficients at α = 0.1:", np.sum(np.abs(coefs_lasso[np.argmin(abs(alphas-0.1))]) > 1e-8), "of 14")

### ✏️ Exercise 5 — Choosing $\alpha$ by cross-validation

`RidgeCV` and `LassoCV` fit the model for a grid of $\alpha$ values and pick the best one by cross-validation on the training set.

1. Fit `poly_model(14, RidgeCV(alphas=alphas))` and `poly_model(14, LassoCV(alphas=alphas, cv=5, max_iter=100_000))` on the small training set. Call them `ridge_cv` and `lasso_cv`.
2. Print the chosen $\alpha$ for each (the regressor is the last pipeline step: `ridge_cv[-1].alpha_`) and the number of non-zero Lasso coefficients.
3. Fill in the dictionary `results` with the test RMSE of: degree-1 OLS, degree-14 OLS, degree-14 Ridge, degree-14 Lasso.

In [ ]:
alphas = np.logspace(-4, 3, 60)
warnings.filterwarnings("ignore", category=ConvergenceWarning)   # Lasso is slow to converge at tiny alpha

### BEGIN SOLUTION
ridge_cv = poly_model(14, RidgeCV(alphas=alphas)).fit(X_tr, y_tr)
lasso_cv = poly_model(14, LassoCV(alphas=alphas, cv=5, max_iter=100_000)).fit(X_tr, y_tr)
### END SOLUTION

print(f"Ridge: chosen alpha = {ridge_cv[-1].alpha_:.3g}")
print(f"Lasso: chosen alpha = {lasso_cv[-1].alpha_:.3g},  non-zero coefficients: "
      f"{np.sum(lasso_cv[-1].coef_ != 0)} of 14")

results = {
    "degree 1,  OLS  ": rmse(y_te, poly_model(1).fit(X_tr, y_tr).predict(X_te)),
    "degree 14, OLS  ": rmse(y_te, poly_model(14).fit(X_tr, y_tr).predict(X_te)),
    "degree 14, Ridge": rmse(y_te, ridge_cv.predict(X_te)),
    "degree 14, Lasso": rmse(y_te, lasso_cv.predict(X_te)),
}

print()
for name, val in results.items():
    print(f"{name}:  test RMSE = {val:10.1f}")

## Summary and outlook

* **Least squares** has a closed-form solution through the normal equations; with known uncertainties, weighted least squares (minimising $\chi^2$) is the right choice and also gives parameter uncertainties.
* **Linear regression** in `scikit-learn` is the same computation; the ML framing shifts the focus from parameters to **predictions** on unseen data.
* The **training error** is always optimistic. Model quality has to be measured on data the model has not seen — a held-out **test set**, and **cross-validation** for any choices we make along the way (degree, $\alpha$, features).
* Flexible models trained on small data sets **overfit**. We can control this with simpler models or with **regularization**: Ridge shrinks coefficients, Lasso also selects features.